# Notebook 3 — Build Gold Layer
**BankGuard | Microsoft Fabric**

## What does this notebook do?
The Gold layer contains **aggregated, ready-to-use tables** that Power BI will connect to directly.

Think of Gold as pre-computing the answers to questions your dashboard will ask:
- "How many transactions happened each month per channel?"
- "What is the NPA ratio by loan type?"
- "Who are our top 10 spending customers?"

We compute these once in Spark, save as Delta tables, and Power BI reads them instantly.

**Run Notebook 2 before this one.**

### Gold tables we'll create:
1. `gold_monthly_transactions` — Spend by month and channel
2. `gold_customer_summary` — One row per customer with key metrics
3. `gold_loan_portfolio` — Loan book summary by NPA category
4. `gold_top_merchants` — Top 10 merchant categories by spend

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

# Read Silver tables
silver_txn   = spark.read.table('silver_transactions')
silver_cust  = spark.read.table('silver_customers')
silver_loans = spark.read.table('silver_loans')

print('Silver tables loaded ✅')

## Gold Table 1: Monthly Transaction Trends

This answers: **"How did transaction volumes and values trend month by month, per channel?"**

Power BI will use this for a line chart showing monthly spend trends.

In [ ]:
# Group by year-month and channel, then count/sum
# This is a classic GROUP BY aggregation

df_monthly = (
    silver_txn
    .groupBy('year_month', 'channel')
    .agg(
        # Count of transactions
        F.count('transaction_id').alias('transaction_count'),

        # Total INR value
        F.round(F.sum('amount'), 2).alias('total_amount'),

        # Average transaction value
        F.round(F.avg('amount'), 2).alias('avg_amount'),

        # How many were declined?
        F.sum(
            F.when(F.col('status') == 'DECLINED', 1).otherwise(0)
        ).alias('declined_count'),

        # How many were international?
        F.sum(
            F.when(F.col('is_international'), 1).otherwise(0)
        ).alias('international_count'),
    )
    .withColumn(
        # Decline rate as a percentage
        'decline_rate_pct',
        F.round(F.col('declined_count') / F.col('transaction_count') * 100, 1)
    )
    .orderBy('year_month', 'channel')
)

df_monthly.write.format('delta').mode('overwrite').saveAsTable('gold_monthly_transactions')
print(f'✅ gold_monthly_transactions: {df_monthly.count():,} rows')
df_monthly.show(8)

## Gold Table 2: Customer Summary

This creates **one row per customer** combining info from all 3 source tables.

Power BI will use this for customer segmentation visuals.

In [ ]:
# Step A: Aggregate transactions per customer
txn_per_customer = (
    silver_txn
    .groupBy('customer_id')
    .agg(
        F.count('transaction_id').alias('total_transactions'),
        F.round(F.sum('amount'), 2).alias('total_spend'),
        F.round(F.avg('amount'), 2).alias('avg_transaction_value'),
        F.round(F.max('amount'), 2).alias('largest_transaction'),
        F.countDistinct('channel').alias('channels_used'),
        F.countDistinct('merchant_category').alias('unique_categories'),
        F.sum(
            F.when(F.col('is_international'), F.col('amount')).otherwise(0)
        ).alias('international_spend'),
    )
)

# Step B: Aggregate loans per customer
loans_per_customer = (
    silver_loans
    .groupBy('customer_id')
    .agg(
        F.count('loan_id').alias('total_loans'),
        F.round(F.sum('outstanding_amount'), 2).alias('total_outstanding'),
        F.max('dpd').alias('max_days_past_due'),
        # Is at least one loan NPA?
        F.max(F.col('is_npa').cast('int')).cast('boolean').alias('has_npa_loan'),
        F.round(F.sum('provision_amount'), 2).alias('total_provisions'),
    )
)

# Step C: Join customer profile + transactions + loans
# LEFT JOIN means: keep all customers, even if they have no transactions/loans
df_customer_summary = (
    silver_cust
    .join(txn_per_customer,   on='customer_id', how='left')
    .join(loans_per_customer, on='customer_id', how='left')
    # Fill null values with 0 for numeric columns
    .fillna(0, subset=['total_transactions','total_spend','total_loans','total_outstanding'])
    .fillna(False, subset=['has_npa_loan'])
)

df_customer_summary.write.format('delta').mode('overwrite').saveAsTable('gold_customer_summary')
print(f'✅ gold_customer_summary: {df_customer_summary.count():,} rows')

df_customer_summary.select(
    'customer_id','segment','age_group','total_transactions',
    'total_spend','total_loans','has_npa_loan'
).show(5)

## Gold Table 3: Loan Portfolio Summary

This answers: **"What does our loan book look like by NPA category and loan type?"**

This is the key table for the regulatory/risk section of the dashboard.

In [ ]:
df_loan_portfolio = (
    silver_loans
    .groupBy('loan_type', 'npa_category')
    .agg(
        F.count('loan_id').alias('loan_count'),
        F.round(F.sum('principal_amount'), 2).alias('total_principal'),
        F.round(F.sum('outstanding_amount'), 2).alias('total_outstanding'),
        F.round(F.sum('provision_amount'), 2).alias('total_provisions'),
        F.round(F.avg('dpd'), 1).alias('avg_days_past_due'),
        F.round(F.avg('interest_rate') * 100, 2).alias('avg_interest_rate_pct'),
    )
    .withColumn(
        # Provision Coverage Ratio = Provisions / Outstanding
        # Higher = more conservative = better
        'provision_coverage_pct',
        F.round(
            F.col('total_provisions') / F.col('total_outstanding') * 100, 1
        )
    )
    .orderBy('loan_type', 'npa_category')
)

df_loan_portfolio.write.format('delta').mode('overwrite').saveAsTable('gold_loan_portfolio')
print(f'✅ gold_loan_portfolio: {df_loan_portfolio.count():,} rows')
df_loan_portfolio.show(10)

## Gold Table 4: Top Merchant Categories

A simple ranking table — which merchant categories do customers spend the most on?

In [ ]:
df_merchants = (
    silver_txn
    .filter(F.col('merchant_category').isNotNull())  # Only POS/card transactions have a merchant
    .groupBy('merchant_category')
    .agg(
        F.count('transaction_id').alias('transaction_count'),
        F.round(F.sum('amount'), 2).alias('total_spend'),
        F.round(F.avg('amount'), 2).alias('avg_spend_per_txn'),
        F.countDistinct('customer_id').alias('unique_customers'),
    )
    .orderBy(F.col('total_spend').desc())
)

df_merchants.write.format('delta').mode('overwrite').saveAsTable('gold_merchant_analysis')
print(f'✅ gold_merchant_analysis: {df_merchants.count():,} rows')
df_merchants.show()

## Final Summary

All 4 Gold tables are ready for Power BI!

In [ ]:
print('Gold Layer Summary')
print('=' * 55)
gold_tables = [
    ('gold_monthly_transactions', 'Monthly trends by channel'),
    ('gold_customer_summary',     'One row per customer'),
    ('gold_loan_portfolio',       'Loan book by NPA category'),
    ('gold_merchant_analysis',    'Top merchant categories'),
]
for table, desc in gold_tables:
    count = spark.sql(f'SELECT COUNT(*) as n FROM {table}').collect()[0]['n']
    print(f'  {table:35s}  {count:5,} rows  ← {desc}')

print()
print('✅ Gold layer complete!')
print('👉 Connect Power BI to this Lakehouse SQL endpoint.')
print('👉 Run Notebook 4 for credit risk scoring.')